# 从零实现 TGN 风格时序图网络：Memory、Mailbox 与 Prequential 评估

本 Notebook 用 PyTorch 基础模块手写时间编码、GRU-like memory updater、temporal neighbor aggregation、memory/mailbox 状态机与链接打分；不调用 `nn.GRU/RNN/LSTM`、PyG、DGL、现成 GNN/Transformer/MHA。

重点不只是一个 `forward`：事件严格按 `(timestamp, sequence)` 排序；每条边必须“先预测、后更新”；负样本和 filtered ranking 只能看到查询时刻以前的事实；memory 必须可 reset、detach，并明确 cold-start 语义。数据是周期性合成事件，只验证实现与在线协议，不代表线上泛化。

参考：[Temporal Graph Networks for Deep Learning on Dynamic Graphs, 2020](https://arxiv.org/abs/2006.10637)。时间编码、动态 embedding 与事件流评估还可对照 [TGAT](https://arxiv.org/abs/2002.07962) 和 [JODIE](https://arxiv.org/abs/1908.01207)。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import copy  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import threading  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 5001  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_digest(payload) -> str:  # 定义本节可复用的核心函数。
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()  # 返回当前分支计算出的结果。

def state_digest50(state: dict[str, torch.Tensor]) -> str:  # 定义本节可复用的核心函数。
    h = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        value = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        h.update(key.encode()); h.update(str(value.dtype).encode())  # 执行当前语句以推进本节示例。
        h.update(str(tuple(value.shape)).encode()); h.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return h.hexdigest()  # 返回当前分支计算出的结果。

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert torch.initial_seed() == SEED  # 用受控断言验证关键不变量。
assert state_digest50({"a": torch.zeros(2)}) != state_digest50({"a": torch.ones(2)})  # 用受控断言验证关键不变量。
assert not any(name in globals() for name in ("torch_geometric", "dgl"))  # 用受控断言验证关键不变量。


## 1. 事件顺序是全序，不是只有 timestamp

生产日志常出现相同 timestamp。这里定义唯一因果键 `key=(timestamp, sequence)`：timestamp 非递减，同一 timestamp 内 sequence 严格递增。`event_id` 全局唯一。模型对事件 $e_k$ 的预测只允许读取 key 小于 `e_k.key` 的状态，预测完成后才写入当前事件。

数据含 8 个 user、4 个活跃 item，以及时间 30 才激活的未来 item。每个 timestamp 有两条事件，时间 1–8/9–12/13–16 分别作为 train/validation/test。切分按时间而不是随机边，避免未来状态倒灌。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Event:  # 定义承载本节状态与行为的数据结构。
    event_id: str  # 执行当前语句以推进本节示例。
    src: int  # 执行当前语句以推进本节示例。
    dst: int  # 执行当前语句以推进本节示例。
    timestamp: int  # 执行当前语句以推进本节示例。
    sequence: int  # 执行当前语句以推进本节示例。
    feature: tuple[float, float]  # 执行当前语句以推进本节示例。
    split: str  # 执行当前语句以推进本节示例。

    @property  # 为下方定义附加声明式配置。
    def key(self) -> tuple[int, int]:  # 定义本节可复用的核心函数。
        return self.timestamp, self.sequence  # 返回当前分支计算出的结果。

NUM_USERS, NUM_ACTIVE_ITEMS, NUM_NODES = 8, 4, 13  # 计算并保存当前步骤的中间状态。
USER_IDS = tuple(range(NUM_USERS))  # 计算并保存当前步骤的中间状态。
ITEM_IDS = tuple(range(8, 13))  # 计算并保存当前步骤的中间状态。
ACTIVE_FROM = {node: (30 if node == 12 else 0) for node in range(NUM_NODES)}  # 计算并保存当前步骤的中间状态。

SPLIT_RANK50 = {"train": 0, "val": 1, "test": 2}  # 计算并保存当前步骤的中间状态。

def validate_key50(key: tuple[int, int], name: str = "key") -> tuple[int, int]:  # 定义本节可复用的核心函数。
    if (not isinstance(key, tuple) or len(key) != 2  # 按当前条件选择后续控制路径。
            or any(not isinstance(value, int) or isinstance(value, bool) or value < 0 for value in key)):  # 执行当前语句以推进本节示例。
        raise ValueError(f"{name} 必须是两个非负整数构成的 tuple")  # 遇到非法合同立即显式失败。
    return key  # 返回当前分支计算出的结果。

def validate_query50(src: int, dst: int, query_key: tuple[int, int]) -> tuple[int, int]:  # 定义本节可复用的核心函数。
    key = validate_key50(query_key, "query_key")  # 计算并保存当前步骤的中间状态。
    if (not isinstance(src, int) or isinstance(src, bool) or src not in USER_IDS  # 按当前条件选择后续控制路径。
            or not isinstance(dst, int) or isinstance(dst, bool) or dst not in ITEM_IDS):  # 执行当前语句以推进本节示例。
        raise ValueError("查询违反 user→item schema")  # 遇到非法合同立即显式失败。
    if ACTIVE_FROM[src] > key[0] or ACTIVE_FROM[dst] > key[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("查询引用尚未激活节点")  # 遇到非法合同立即显式失败。
    return key  # 返回当前分支计算出的结果。

def validate_event50(event: Event) -> tuple[int, int]:  # 定义本节可复用的核心函数。
    if not isinstance(event, Event):  # 按当前条件选择后续控制路径。
        raise ValueError("事件必须是 Event")  # 遇到非法合同立即显式失败。
    key = validate_query50(event.src, event.dst, event.key)  # 计算并保存当前步骤的中间状态。
    if not isinstance(event.event_id, str) or not event.event_id.strip():  # 按当前条件选择后续控制路径。
        raise ValueError("event_id 必须是非空字符串")  # 遇到非法合同立即显式失败。
    if event.split not in SPLIT_RANK50:  # 按当前条件选择后续控制路径。
        raise ValueError("split 非法")  # 遇到非法合同立即显式失败。
    if (not isinstance(event.feature, tuple) or len(event.feature) != 2  # 按当前条件选择后续控制路径。
            or any(isinstance(value, bool) or not isinstance(value, (int, float))  # 执行当前语句以推进本节示例。
                   or not math.isfinite(float(value)) for value in event.feature)):  # 执行当前语句以推进本节示例。
        raise ValueError("事件 feature 必须是两个有限数构成的 tuple")  # 遇到非法合同立即显式失败。
    return key  # 返回当前分支计算出的结果。

def validate_event_stream(events: list[Event]) -> None:  # 定义本节可复用的核心函数。
    if not isinstance(events, list) or not events:  # 按当前条件选择后续控制路径。
        raise ValueError("事件流必须是非空 list")  # 遇到非法合同立即显式失败。
    previous, previous_split_rank, event_ids = None, -1, set()  # 计算并保存当前步骤的中间状态。
    for event in events:  # 遍历输入元素以累积或检查结果。
        key = validate_event50(event)  # 计算并保存当前步骤的中间状态。
        if event.event_id in event_ids:  # 按当前条件选择后续控制路径。
            raise ValueError("event_id 必须唯一")  # 遇到非法合同立即显式失败。
        if SPLIT_RANK50[event.split] < previous_split_rank:  # 按当前条件选择后续控制路径。
            raise ValueError("split 阶段不能随时间倒退")  # 遇到非法合同立即显式失败。
        if previous is not None and key <= previous:  # 按当前条件选择后续控制路径。
            raise ValueError("事件必须按 (timestamp, sequence) 严格排序")  # 遇到非法合同立即显式失败。
        event_ids.add(event.event_id)  # 执行当前语句以推进本节示例。
        previous, previous_split_rank = key, SPLIT_RANK50[event.split]  # 计算并保存当前步骤的中间状态。

events50 = []  # 计算并保存当前步骤的中间状态。
for timestamp in range(1, 17):  # 遍历输入元素以累积或检查结果。
    split = "train" if timestamp <= 8 else ("val" if timestamp <= 12 else "test")  # 计算并保存当前步骤的中间状态。
    for sequence in range(2):  # 遍历输入元素以累积或检查结果。
        index = 2 * (timestamp - 1) + sequence  # 计算并保存当前步骤的中间状态。
        src = index % NUM_USERS  # 计算并保存当前步骤的中间状态。
        dst = 8 + src % NUM_ACTIVE_ITEMS  # 计算并保存当前步骤的中间状态。
        events50.append(Event(f"e-{timestamp:02d}-{sequence}", src, dst, timestamp, sequence,  # 执行当前语句以推进本节示例。
                              (1.0, float(sequence)), split))  # 执行当前语句以推进本节示例。
validate_event_stream(events50)  # 执行当前语句以推进本节示例。
split_events50 = {s: [e for e in events50 if e.split == s] for s in ("train", "val", "test")}  # 计算并保存当前步骤的中间状态。
assert tuple(len(split_events50[s]) for s in ("train", "val", "test")) == (16, 8, 8)  # 用受控断言验证关键不变量。
assert max(e.key for e in split_events50["train"]) < min(e.key for e in split_events50["val"])  # 用受控断言验证关键不变量。
assert max(e.key for e in split_events50["val"]) < min(e.key for e in split_events50["test"])  # 用受控断言验证关键不变量。
assert all(e.dst == 8 + e.src % 4 for e in events50)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    validate_event_stream([events50[1], events50[0]])  # 执行当前语句以推进本节示例。
    raise AssertionError("相同 timestamp 的逆 sequence 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "严格排序" in str(exc)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    validate_event_stream([Event("bad-feature", 0, 8, 1, 0, (float("nan"), 0.0), "train")])  # 执行当前语句以推进本节示例。
    raise AssertionError("非有限事件特征未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "feature" in str(exc)  # 用受控断言验证关键不变量。


## 2. 时间编码与手写 GRU-like updater

时间差编码使用可学习频率：$\phi(\Delta t)=\cos(\omega\Delta t+b)$。负时间差意味着读取未来，必须拒绝。

更新器显式实现

$$z=\sigma(W_zx+U_zh),\quad r=\sigma(W_rx+U_rh),$$
$$n=\tanh(W_nx+U_n(r\odot h)),\quad h'=(1-z)\odot n+z\odot h.$$

它是普通 `nn.Module`，没有导入循环网络层；这样门控公式、shape 与 detach 边界都可直接审计。


In [ ]:
class TimeEncoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, time_dim: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if time_dim <= 0: raise ValueError("time_dim 必须为正")  # 按当前条件选择后续控制路径。
        freq = torch.logspace(0, -2, time_dim)  # 计算并保存当前步骤的中间状态。
        self.frequency = nn.Parameter(freq)  # 计算并保存当前步骤的中间状态。
        self.phase = nn.Parameter(torch.zeros(time_dim))  # 计算并保存当前步骤的中间状态。

    def forward(self, delta: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if delta.ndim != 1 or not torch.is_floating_point(delta) or not torch.isfinite(delta).all():  # 按当前条件选择后续控制路径。
            raise ValueError("delta 必须是有限浮点一维张量")  # 遇到非法合同立即显式失败。
        if bool((delta < 0).any()): raise ValueError("负时间差意味着未来泄漏")  # 按当前条件选择后续控制路径。
        encoded = torch.cos(delta[:, None] * self.frequency[None, :] + self.phase[None, :])  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(encoded).all()): raise ValueError("时间编码参数产生非有限值")  # 按当前条件选择后续控制路径。
        return encoded  # 返回当前分支计算出的结果。

class ManualGRUCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim: int, hidden_dim: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if input_dim <= 0 or hidden_dim <= 0: raise ValueError("GRU-like 维度必须为正")  # 按当前条件选择后续控制路径。
        self.input_dim, self.hidden_dim = input_dim, hidden_dim  # 计算并保存当前步骤的中间状态。
        self.x_z = nn.Linear(input_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.h_z = nn.Linear(hidden_dim, hidden_dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.x_r = nn.Linear(input_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.h_r = nn.Linear(hidden_dim, hidden_dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.x_n = nn.Linear(input_dim, hidden_dim)  # 计算并保存当前步骤的中间状态。
        self.h_n = nn.Linear(hidden_dim, hidden_dim, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, hidden: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x.shape[:-1] != hidden.shape[:-1] or x.shape[-1] != self.input_dim or hidden.shape[-1] != self.hidden_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("updater 输入 shape 不匹配")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(x).all() or not torch.isfinite(hidden).all():  # 按当前条件选择后续控制路径。
            raise ValueError("updater 输入含非有限值")  # 遇到非法合同立即显式失败。
        z = torch.sigmoid(self.x_z(x) + self.h_z(hidden))  # 计算并保存当前步骤的中间状态。
        r = torch.sigmoid(self.x_r(x) + self.h_r(hidden))  # 计算并保存当前步骤的中间状态。
        candidate = torch.tanh(self.x_n(x) + self.h_n(r * hidden))  # 计算并保存当前步骤的中间状态。
        updated = (1.0 - z) * candidate + z * hidden  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(updated).all()): raise ValueError("updater 参数产生非有限值")  # 按当前条件选择后续控制路径。
        return updated  # 返回当前分支计算出的结果。

time_probe50 = TimeEncoder(4)(torch.tensor([0.0, 2.0]))  # 计算并保存当前步骤的中间状态。
assert time_probe50.shape == (2, 4) and torch.allclose(time_probe50[0], torch.ones(4))  # 用受控断言验证关键不变量。
cell_probe50 = ManualGRUCell(3, 2)  # 计算并保存当前步骤的中间状态。
for parameter in cell_probe50.parameters(): nn.init.zeros_(parameter)  # 遍历输入元素以累积或检查结果。
probe_hidden50 = torch.tensor([[2.0, -2.0]], requires_grad=True)  # 计算并保存当前步骤的中间状态。
probe_new50 = cell_probe50(torch.ones(1, 3), probe_hidden50)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(probe_new50, 0.5 * probe_hidden50)  # 用受控断言验证关键不变量。
probe_new50.sum().backward()  # 执行当前语句以推进本节示例。
assert torch.allclose(probe_hidden50.grad, torch.full_like(probe_hidden50, 0.5))  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    TimeEncoder(2)(torch.tensor([-1.0]))  # 执行当前语句以推进本节示例。
    raise AssertionError("未来时间差未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "未来泄漏" in str(exc)  # 用受控断言验证关键不变量。


## 3. MemoryBank 与 Mailbox 的因果合同

`memory[node]` 是截至上一事件的压缩状态；mailbox 保存最近若干条 `(event_key, neighbor_memory_snapshot, edge_feature)`。保存当时的 neighbor snapshot，而不是查询时的“最新 neighbor memory”，避免经由共享邻居间接读取未来。

`read/context(node, query_key)` 要求已写入的最后 key 严格小于 query key，并返回 tensor 深拷贝；调用方无法借修改返回值污染内部状态。若先更新当前事件再预测，立即 fail-closed。写入同时验证 key、CPU float32、shape、feature 维度和所有数值有限性，并对 updater 输出 `detach().clone()`，阻断跨无限事件流的 autograd 图。双端点通过 `commit_many` 先全部验证再写入；每轮训练和独立评估前必须 `reset()`。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Mail:  # 定义承载本节状态与行为的数据结构。
    key: tuple[int, int]  # 执行当前语句以推进本节示例。
    neighbor_memory: torch.Tensor  # 执行当前语句以推进本节示例。
    feature: torch.Tensor  # 执行当前语句以推进本节示例。

class MemoryBank:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, num_nodes: int, memory_dim: int, feature_dim: int = 2, mailbox_size: int = 4):  # 定义本节可复用的核心函数。
        values = (num_nodes, memory_dim, feature_dim, mailbox_size)  # 计算并保存当前步骤的中间状态。
        if any(not isinstance(value, int) or isinstance(value, bool) or value <= 0 for value in values):  # 按当前条件选择后续控制路径。
            raise ValueError("MemoryBank 配置必须是正整数")  # 遇到非法合同立即显式失败。
        self.num_nodes, self.memory_dim = num_nodes, memory_dim  # 计算并保存当前步骤的中间状态。
        self.feature_dim, self.mailbox_size = feature_dim, mailbox_size  # 计算并保存当前步骤的中间状态。
        self.reset()  # 执行当前语句以推进本节示例。

    @staticmethod  # 为下方定义附加声明式配置。
    def _clone_mail(mail: Mail) -> Mail:  # 定义本节可复用的核心函数。
        return Mail(tuple(mail.key), mail.neighbor_memory.detach().clone(), mail.feature.detach().clone())  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def memory(self) -> torch.Tensor:  # 定义本节可复用的核心函数。
        return self._memory.detach().clone()  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def last_key(self) -> list[tuple[int, int] | None]:  # 定义本节可复用的核心函数。
        return list(self._last_key)  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def mailbox(self) -> list[list[Mail]]:  # 定义本节可复用的核心函数。
        return [[self._clone_mail(mail) for mail in box] for box in self._mailbox]  # 返回当前分支计算出的结果。

    def reset(self) -> None:  # 定义本节可复用的核心函数。
        self._memory = torch.zeros(self.num_nodes, self.memory_dim, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        self._last_key: list[tuple[int, int] | None] = [None] * self.num_nodes  # 计算并保存当前步骤的中间状态。
        self._mailbox: list[list[Mail]] = [[] for _ in range(self.num_nodes)]  # 计算并保存当前步骤的中间状态。

    def _node(self, node: int) -> None:  # 定义本节可复用的核心函数。
        if not isinstance(node, int) or isinstance(node, bool) or not (0 <= node < self.num_nodes):  # 按当前条件选择后续控制路径。
            raise ValueError("节点编号越界或类型非法")  # 遇到非法合同立即显式失败。

    def _vector(self, value: torch.Tensor, width: int, name: str) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if (not isinstance(value, torch.Tensor) or value.shape != (width,)  # 按当前条件选择后续控制路径。
                or not torch.is_floating_point(value) or value.device != self._memory.device  # 计算并保存当前步骤的中间状态。
                or value.dtype != self._memory.dtype or not bool(torch.isfinite(value).all())):  # 计算并保存当前步骤的中间状态。
            raise ValueError(f"{name} 必须是 CPU float32 有限向量 [{width}]")  # 遇到非法合同立即显式失败。
        return value.detach().clone()  # 返回当前分支计算出的结果。

    @staticmethod  # 为下方定义附加声明式配置。
    def _payload_key(raw, name: str) -> tuple[int, int]:  # 定义本节可复用的核心函数。
        if not isinstance(raw, list): raise ValueError(f"{name} 必须是 JSON list")  # 按当前条件选择后续控制路径。
        return validate_key50(tuple(raw), name)  # 返回当前分支计算出的结果。

    def read(self, node: int, query_key: tuple[int, int]) -> torch.Tensor:  # 定义本节可复用的核心函数。
        self._node(node); key = validate_key50(query_key, "query_key")  # 计算并保存当前步骤的中间状态。
        if self._last_key[node] is not None and self._last_key[node] >= key:  # 按当前条件选择后续控制路径。
            raise ValueError("memory 含当前或未来事件，违反先预测后更新")  # 遇到非法合同立即显式失败。
        return self._memory[node].detach().clone()  # 返回当前分支计算出的结果。

    def context(self, node: int, query_key: tuple[int, int]) -> list[Mail]:  # 定义本节可复用的核心函数。
        self._node(node); key = validate_key50(query_key, "query_key")  # 计算并保存当前步骤的中间状态。
        if self._last_key[node] is not None and self._last_key[node] >= key:  # 按当前条件选择后续控制路径。
            raise ValueError("mailbox 含当前或未来事件")  # 遇到非法合同立即显式失败。
        selected = self._mailbox[node][-self.mailbox_size:]  # 计算并保存当前步骤的中间状态。
        if any(mail.key >= key for mail in selected):  # 按当前条件选择后续控制路径。
            raise ValueError("mailbox 含当前或未来事件")  # 遇到非法合同立即显式失败。
        return [self._clone_mail(mail) for mail in selected]  # 返回当前分支计算出的结果。

    def _validated_update(self, node: int, key: tuple[int, int], new_memory: torch.Tensor,  # 定义本节可复用的核心函数。
                          mail: Mail) -> tuple[int, tuple[int, int], torch.Tensor, Mail]:  # 执行当前语句以推进本节示例。
        self._node(node); key = validate_key50(key, "commit key")  # 计算并保存当前步骤的中间状态。
        if self._last_key[node] is not None and key <= self._last_key[node]:  # 按当前条件选择后续控制路径。
            raise ValueError("memory 更新 key 未严格递增")  # 遇到非法合同立即显式失败。
        if not isinstance(mail, Mail):  # 按当前条件选择后续控制路径。
            raise ValueError("mail 必须是 Mail")  # 遇到非法合同立即显式失败。
        mail_key = validate_key50(mail.key, "mail key")  # 计算并保存当前步骤的中间状态。
        if mail_key != key:  # 按当前条件选择后续控制路径。
            raise ValueError("mail.key 与 commit key 不一致")  # 遇到非法合同立即显式失败。
        memory_copy = self._vector(new_memory, self.memory_dim, "new_memory")  # 计算并保存当前步骤的中间状态。
        neighbor_copy = self._vector(mail.neighbor_memory, self.memory_dim, "neighbor_memory")  # 计算并保存当前步骤的中间状态。
        feature_copy = self._vector(mail.feature, self.feature_dim, "mail.feature")  # 计算并保存当前步骤的中间状态。
        return node, key, memory_copy, Mail(key, neighbor_copy, feature_copy)  # 返回当前分支计算出的结果。

    def commit_many(self, updates: list[tuple[int, tuple[int, int], torch.Tensor, Mail]]) -> None:  # 定义本节可复用的核心函数。
        if not isinstance(updates, list) or not updates:  # 按当前条件选择后续控制路径。
            raise ValueError("updates 必须是非空 list")  # 遇到非法合同立即显式失败。
        validated = [self._validated_update(*update) for update in updates]  # 计算并保存当前步骤的中间状态。
        nodes = [update[0] for update in validated]  # 计算并保存当前步骤的中间状态。
        if len(nodes) != len(set(nodes)):  # 按当前条件选择后续控制路径。
            raise ValueError("一次原子提交不能重复更新同一节点")  # 遇到非法合同立即显式失败。
        # 上面所有项目验证成功后才进入写阶段，因此第二个端点失败不会留下半次边更新。
        for node, key, memory_copy, mail_copy in validated:  # 遍历输入元素以累积或检查结果。
            self._memory[node] = memory_copy  # 计算并保存当前步骤的中间状态。
            self._last_key[node] = key  # 计算并保存当前步骤的中间状态。
            self._mailbox[node].append(mail_copy)  # 执行当前语句以推进本节示例。
            self._mailbox[node] = self._mailbox[node][-self.mailbox_size:]  # 计算并保存当前步骤的中间状态。

    def commit(self, node: int, key: tuple[int, int], new_memory: torch.Tensor, mail: Mail) -> None:  # 定义本节可复用的核心函数。
        self.commit_many([(node, key, new_memory, mail)])  # 执行当前语句以推进本节示例。

    def clone(self) -> "MemoryBank":  # 定义本节可复用的核心函数。
        cloned = MemoryBank(self.num_nodes, self.memory_dim, self.feature_dim, self.mailbox_size)  # 计算并保存当前步骤的中间状态。
        cloned._memory = self._memory.detach().clone()  # 计算并保存当前步骤的中间状态。
        cloned._last_key = list(self._last_key)  # 计算并保存当前步骤的中间状态。
        cloned._mailbox = [[self._clone_mail(mail) for mail in box] for box in self._mailbox]  # 计算并保存当前步骤的中间状态。
        return cloned  # 返回当前分支计算出的结果。

    def state_payload(self) -> dict:  # 定义本节可复用的核心函数。
        return {  # 返回当前分支计算出的结果。
            "config": {"num_nodes": self.num_nodes, "memory_dim": self.memory_dim,  # 执行当前语句以推进本节示例。
                       "feature_dim": self.feature_dim, "mailbox_size": self.mailbox_size},  # 执行当前语句以推进本节示例。
            "memory": self._memory.detach().cpu().tolist(),  # 执行当前语句以推进本节示例。
            "last_key": [None if key is None else list(key) for key in self._last_key],  # 执行当前语句以推进本节示例。
            "mailbox": [[{"key": list(mail.key), "neighbor_memory": mail.neighbor_memory.tolist(),  # 执行当前语句以推进本节示例。
                           "feature": mail.feature.tolist()} for mail in box] for box in self._mailbox],  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。

    @classmethod  # 为下方定义附加声明式配置。
    def from_payload(cls, payload: dict) -> "MemoryBank":  # 定义本节可复用的核心函数。
        if not isinstance(payload, dict) or set(payload) != {"config", "memory", "last_key", "mailbox"}:  # 按当前条件选择后续控制路径。
            raise ValueError("MemoryBank snapshot 字段非法")  # 遇到非法合同立即显式失败。
        config = payload["config"]  # 计算并保存当前步骤的中间状态。
        if not isinstance(config, dict) or set(config) != {"num_nodes", "memory_dim", "feature_dim", "mailbox_size"}:  # 按当前条件选择后续控制路径。
            raise ValueError("MemoryBank snapshot config 非法")  # 遇到非法合同立即显式失败。
        bank = cls(**config)  # 计算并保存当前步骤的中间状态。
        try:  # 尝试执行可能失败的受控操作。
            memory = torch.tensor(payload["memory"], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        except (TypeError, ValueError) as exc:  # 捕获预期异常并验证失败分支。
            raise ValueError("snapshot memory 不能解码") from exc  # 遇到非法合同立即显式失败。
        if memory.shape != (bank.num_nodes, bank.memory_dim) or not bool(torch.isfinite(memory).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("snapshot memory shape/数值非法")  # 遇到非法合同立即显式失败。
        if (not isinstance(payload["last_key"], list) or len(payload["last_key"]) != bank.num_nodes  # 按当前条件选择后续控制路径。
                or not isinstance(payload["mailbox"], list) or len(payload["mailbox"]) != bank.num_nodes):  # 计算并保存当前步骤的中间状态。
            raise ValueError("snapshot 节点状态数量非法")  # 遇到非法合同立即显式失败。
        last_keys, boxes = [], []  # 计算并保存当前步骤的中间状态。
        for node, (raw_last, raw_box) in enumerate(zip(payload["last_key"], payload["mailbox"])):  # 遍历输入元素以累积或检查结果。
            last = None if raw_last is None else bank._payload_key(raw_last, "snapshot last_key")  # 计算并保存当前步骤的中间状态。
            if not isinstance(raw_box, list) or len(raw_box) > bank.mailbox_size:  # 按当前条件选择后续控制路径。
                raise ValueError("snapshot mailbox 长度非法")  # 遇到非法合同立即显式失败。
            box = []  # 计算并保存当前步骤的中间状态。
            for raw_mail in raw_box:  # 遍历输入元素以累积或检查结果。
                if not isinstance(raw_mail, dict) or set(raw_mail) != {"key", "neighbor_memory", "feature"}:  # 按当前条件选择后续控制路径。
                    raise ValueError("snapshot mail 字段非法")  # 遇到非法合同立即显式失败。
                key = bank._payload_key(raw_mail["key"], "snapshot mail key")  # 计算并保存当前步骤的中间状态。
                try:  # 尝试执行可能失败的受控操作。
                    neighbor = torch.tensor(raw_mail["neighbor_memory"], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
                    feature = torch.tensor(raw_mail["feature"], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
                except (TypeError, ValueError, RuntimeError) as exc:  # 捕获预期异常并验证失败分支。
                    raise ValueError("snapshot mail tensor 不能解码") from exc  # 遇到非法合同立即显式失败。
                box.append(Mail(key, bank._vector(neighbor, bank.memory_dim, "snapshot neighbor"),  # 执行当前语句以推进本节示例。
                                     bank._vector(feature, bank.feature_dim, "snapshot feature")))  # 执行当前语句以推进本节示例。
            if any(box[index].key >= box[index + 1].key for index in range(len(box) - 1)):  # 按当前条件选择后续控制路径。
                raise ValueError("snapshot mailbox key 未严格递增")  # 遇到非法合同立即显式失败。
            if (last is None) != (len(box) == 0) or (box and box[-1].key != last):  # 按当前条件选择后续控制路径。
                raise ValueError("snapshot last_key 与 mailbox 不一致")  # 遇到非法合同立即显式失败。
            last_keys.append(last); boxes.append(box)  # 执行当前语句以推进本节示例。
        bank._memory, bank._last_key, bank._mailbox = memory, last_keys, boxes  # 计算并保存当前步骤的中间状态。
        return bank  # 返回当前分支计算出的结果。

bank_probe50 = MemoryBank(3, 4, feature_dim=2, mailbox_size=2)  # 计算并保存当前步骤的中间状态。
assert torch.count_nonzero(bank_probe50.memory) == 0  # 用受控断言验证关键不变量。
assert all(len(box) == 0 for box in bank_probe50.mailbox)  # 用受控断言验证关键不变量。
assert bank_probe50.read(0, (1, 0)).shape == (4,)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    bank_probe50.commit_many([  # 执行当前语句以推进本节示例。
        (0, (1, 0), torch.ones(4), Mail((1, 0), torch.zeros(4), torch.zeros(2))),  # 执行当前语句以推进本节示例。
        (1, (1, 0), torch.ones(4), Mail((1, 0), torch.full((4,), float("nan")), torch.zeros(2))),  # 执行当前语句以推进本节示例。
    ])  # 执行当前语句以推进本节示例。
    raise AssertionError("非法第二端点导致半提交")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert bank_probe50.last_key == [None, None, None]  # 用受控断言验证关键不变量。


## 4. Temporal neighbor aggregation

对 mailbox 中每条历史消息，把 neighbor memory、edge feature 与 $\phi(t-t_e)$ 拼接，投影为 message，再用手写标量 attention 做加权和。空 mailbox 返回精确零向量；所有历史 key 必须小于查询 key，所以时间差非负。

设 mailbox 上限为 $K$、memory 维度 $D$，单节点查询成本约 $O(KD^2)$，状态内存约 $O(ND+NK(D+F))$。生产系统通常用按时间排序的邻居索引和批量 scatter kernel。


In [ ]:
class TemporalNeighborAggregator(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, memory_dim: int, feature_dim: int, time_dim: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.memory_dim, self.feature_dim = memory_dim, feature_dim  # 计算并保存当前步骤的中间状态。
        self.time_encoder = TimeEncoder(time_dim)  # 计算并保存当前步骤的中间状态。
        self.message = nn.Linear(memory_dim + feature_dim + time_dim, memory_dim)  # 计算并保存当前步骤的中间状态。
        self.score = nn.Linear(memory_dim, 1, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, neighbor_memory: torch.Tensor, edge_feature: torch.Tensor,  # 定义本节可复用的核心函数。
                delta: torch.Tensor) -> torch.Tensor:  # 执行当前语句以推进本节示例。
        if (not isinstance(neighbor_memory, torch.Tensor) or neighbor_memory.ndim != 2  # 按当前条件选择后续控制路径。
                or neighbor_memory.shape[1] != self.memory_dim or not torch.is_floating_point(neighbor_memory)):  # 计算并保存当前步骤的中间状态。
            raise ValueError("neighbor_memory 必须是浮点 [K,D]")  # 遇到非法合同立即显式失败。
        k = neighbor_memory.shape[0]  # 计算并保存当前步骤的中间状态。
        if (not isinstance(edge_feature, torch.Tensor) or not isinstance(delta, torch.Tensor)  # 按当前条件选择后续控制路径。
                or edge_feature.shape != (k, self.feature_dim) or delta.shape != (k,)  # 计算并保存当前步骤的中间状态。
                or not torch.is_floating_point(edge_feature) or not torch.is_floating_point(delta)):  # 执行当前语句以推进本节示例。
            raise ValueError("mailbox feature/delta shape 不匹配")  # 遇到非法合同立即显式失败。
        if (neighbor_memory.device != edge_feature.device or neighbor_memory.device != delta.device  # 按当前条件选择后续控制路径。
                or neighbor_memory.dtype != edge_feature.dtype or neighbor_memory.dtype != delta.dtype  # 计算并保存当前步骤的中间状态。
                or neighbor_memory.device != self.message.weight.device  # 计算并保存当前步骤的中间状态。
                or neighbor_memory.dtype != self.message.weight.dtype  # 计算并保存当前步骤的中间状态。
                or not bool(torch.isfinite(neighbor_memory).all())  # 执行当前语句以推进本节示例。
                or not bool(torch.isfinite(edge_feature).all()) or not bool(torch.isfinite(delta).all())):  # 执行当前语句以推进本节示例。
            raise ValueError("聚合输入必须同 device/dtype 且全部有限")  # 遇到非法合同立即显式失败。
        if k == 0:  # 按当前条件选择后续控制路径。
            return neighbor_memory.new_zeros(self.memory_dim)  # 返回当前分支计算出的结果。
        encoded_time = self.time_encoder(delta)  # 计算并保存当前步骤的中间状态。
        message = torch.tanh(self.message(torch.cat([neighbor_memory, edge_feature, encoded_time], dim=-1)))  # 计算并保存当前步骤的中间状态。
        logits = self.score(message).squeeze(-1)  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(message).all()) or not bool(torch.isfinite(logits).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("聚合器参数或中间值产生非有限数")  # 遇到非法合同立即显式失败。
        shifted = logits - logits.max()  # 计算并保存当前步骤的中间状态。
        numerator, denominator = shifted.exp(), shifted.exp().sum()  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(denominator)) or float(denominator) <= 0.0:  # 按当前条件选择后续控制路径。
            raise ValueError("attention 归一化失败")  # 遇到非法合同立即显式失败。
        result = (numerator[:, None] / denominator * message).sum(0)  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(result).all()): raise ValueError("聚合结果非有限")  # 按当前条件选择后续控制路径。
        return result  # 返回当前分支计算出的结果。

agg_probe50 = TemporalNeighborAggregator(4, 2, 3)  # 计算并保存当前步骤的中间状态。
empty_agg50 = agg_probe50(torch.empty(0, 4), torch.empty(0, 2), torch.empty(0))  # 计算并保存当前步骤的中间状态。
assert torch.equal(empty_agg50, torch.zeros(4))  # 用受控断言验证关键不变量。
full_agg50 = agg_probe50(torch.randn(2, 4), torch.ones(2, 2), torch.tensor([1.0, 2.0]))  # 计算并保存当前步骤的中间状态。
assert full_agg50.shape == (4,) and torch.isfinite(full_agg50).all()  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    agg_probe50(torch.full((1, 4), float("inf")), torch.ones(1, 2), torch.ones(1))  # 执行当前语句以推进本节示例。
    raise AssertionError("非有限 neighbor_memory 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "有限" in str(exc)  # 用受控断言验证关键不变量。


## 5. TGN 链接预测器：读取、打分、再提交

节点表示由静态 embedding、当前 memory 和 temporal neighborhood 聚合组成。`forward(bank,src,dst,key)` 只读状态并返回 logit；`update_after` 才用当前边生成双向消息，经手写 updater 写回 src/dst。

把 update 藏在 `forward` 中会导致负候选的评分顺序影响状态，也无法保证正例没提前写入。本实现严格拆开纯读取打分和有副作用提交；公开打分统一检查 user→item schema、节点激活时刻、bank 配置以及输入/中间/输出的有限性。


In [ ]:
class TGNLinkPredictor(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, num_nodes: int, memory_dim: int = 8, feature_dim: int = 2, time_dim: int = 4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.num_nodes, self.memory_dim, self.feature_dim, self.time_dim = num_nodes, memory_dim, feature_dim, time_dim  # 计算并保存当前步骤的中间状态。
        self.node_embedding = nn.Embedding(num_nodes, memory_dim)  # 计算并保存当前步骤的中间状态。
        self.aggregator = TemporalNeighborAggregator(memory_dim, feature_dim, time_dim)  # 计算并保存当前步骤的中间状态。
        self.node_projection = nn.Linear(3 * memory_dim, memory_dim)  # 计算并保存当前步骤的中间状态。
        self.update_time = TimeEncoder(time_dim)  # 计算并保存当前步骤的中间状态。
        self.update_message = nn.Linear(2 * memory_dim + feature_dim + time_dim, memory_dim)  # 计算并保存当前步骤的中间状态。
        self.updater = ManualGRUCell(memory_dim, memory_dim)  # 计算并保存当前步骤的中间状态。
        self.scorer = nn.Sequential(nn.Linear(3 * memory_dim, memory_dim), nn.ReLU(), nn.Linear(memory_dim, 1))  # 计算并保存当前步骤的中间状态。

    def _validate_bank(self, bank: MemoryBank) -> None:  # 定义本节可复用的核心函数。
        if (not isinstance(bank, MemoryBank) or bank.num_nodes != self.num_nodes  # 按当前条件选择后续控制路径。
                or bank.memory_dim != self.memory_dim or bank.feature_dim != self.feature_dim):  # 计算并保存当前步骤的中间状态。
            raise ValueError("模型与 MemoryBank 配置不一致")  # 遇到非法合同立即显式失败。

    def encode_node(self, bank: MemoryBank, node: int, query_key: tuple[int, int]) -> torch.Tensor:  # 定义本节可复用的核心函数。
        self._validate_bank(bank); validate_key50(query_key, "query_key")  # 执行当前语句以推进本节示例。
        memory = bank.read(node, query_key)  # 计算并保存当前步骤的中间状态。
        mails = bank.context(node, query_key)  # 计算并保存当前步骤的中间状态。
        if mails:  # 按当前条件选择后续控制路径。
            neighbor = torch.stack([m.neighbor_memory for m in mails])  # 计算并保存当前步骤的中间状态。
            feature = torch.stack([m.feature for m in mails])  # 计算并保存当前步骤的中间状态。
            delta = torch.tensor([query_key[0] - m.key[0] for m in mails], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            neighbor = torch.empty(0, self.memory_dim)  # 计算并保存当前步骤的中间状态。
            feature = torch.empty(0, self.feature_dim)  # 计算并保存当前步骤的中间状态。
            delta = torch.empty(0)  # 计算并保存当前步骤的中间状态。
        temporal = self.aggregator(neighbor, feature, delta)  # 计算并保存当前步骤的中间状态。
        return self._compose_node(node, memory, temporal)  # 返回当前分支计算出的结果。

    def _compose_node(self, node: int, memory: torch.Tensor, temporal: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if (not isinstance(node, int) or isinstance(node, bool) or not 0 <= node < self.num_nodes  # 按当前条件选择后续控制路径。
                or not isinstance(memory, torch.Tensor) or not isinstance(temporal, torch.Tensor)  # 执行当前语句以推进本节示例。
                or memory.shape != (self.memory_dim,) or temporal.shape != (self.memory_dim,)  # 计算并保存当前步骤的中间状态。
                or not torch.is_floating_point(memory) or not torch.is_floating_point(temporal)  # 执行当前语句以推进本节示例。
                or memory.device != self.node_embedding.weight.device or temporal.device != memory.device  # 计算并保存当前步骤的中间状态。
                or memory.dtype != self.node_embedding.weight.dtype or temporal.dtype != memory.dtype  # 计算并保存当前步骤的中间状态。
                or not bool(torch.isfinite(memory).all()) or not bool(torch.isfinite(temporal).all())):  # 执行当前语句以推进本节示例。
            raise ValueError("memory/temporal override shape 非法")  # 遇到非法合同立即显式失败。
        static = self.node_embedding(torch.tensor(node))  # 计算并保存当前步骤的中间状态。
        result = torch.tanh(self.node_projection(torch.cat([static, memory, temporal])))  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(result).all()): raise ValueError("节点编码产生非有限值")  # 按当前条件选择后续控制路径。
        return result  # 返回当前分支计算出的结果。

    def _pair_logit(self, z_src: torch.Tensor, z_dst: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if (z_src.shape != (self.memory_dim,) or z_dst.shape != (self.memory_dim,)  # 按当前条件选择后续控制路径。
                or not bool(torch.isfinite(z_src).all()) or not bool(torch.isfinite(z_dst).all())):  # 执行当前语句以推进本节示例。
            raise ValueError("打分表示 shape/数值非法")  # 遇到非法合同立即显式失败。
        logit = self.scorer(torch.cat([z_src, z_dst, z_src * z_dst])).squeeze(-1)  # 计算并保存当前步骤的中间状态。
        if logit.ndim != 0 or not bool(torch.isfinite(logit)):  # 按当前条件选择后续控制路径。
            raise ValueError("链接分数非有限")  # 遇到非法合同立即显式失败。
        return logit  # 返回当前分支计算出的结果。

    def forward(self, bank: MemoryBank, src: int, dst: int, query_key: tuple[int, int]) -> torch.Tensor:  # 定义本节可复用的核心函数。
        self._validate_bank(bank); validate_query50(src, dst, query_key)  # 执行当前语句以推进本节示例。
        z_src = self.encode_node(bank, src, query_key)  # 计算并保存当前步骤的中间状态。
        z_dst = self.encode_node(bank, dst, query_key)  # 计算并保存当前步骤的中间状态。
        return self._pair_logit(z_src, z_dst)  # 返回当前分支计算出的结果。

    def score_memory_states(self, src: int, dst: int, src_memory: torch.Tensor,  # 定义本节可复用的核心函数。
                            dst_memory: torch.Tensor) -> torch.Tensor:  # 执行当前语句以推进本节示例。
        if src not in USER_IDS or dst not in ITEM_IDS:  # 按当前条件选择后续控制路径。
            raise ValueError("打分违反 user→item schema")  # 遇到非法合同立即显式失败。
        if not isinstance(src_memory, torch.Tensor) or not isinstance(dst_memory, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("memory override 必须是 Tensor")  # 遇到非法合同立即显式失败。
        zero = src_memory.new_zeros(self.memory_dim)  # 计算并保存当前步骤的中间状态。
        return self._pair_logit(self._compose_node(src, src_memory, zero),  # 返回当前分支计算出的结果。
                                self._compose_node(dst, dst_memory, zero))  # 执行当前语句以推进本节示例。

    def _update_input(self, own: torch.Tensor, other: torch.Tensor, feature: torch.Tensor, delta: float) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if (own.shape != (self.memory_dim,) or other.shape != (self.memory_dim,)  # 按当前条件选择后续控制路径。
                or feature.shape != (self.feature_dim,) or not bool(torch.isfinite(own).all())  # 计算并保存当前步骤的中间状态。
                or not bool(torch.isfinite(other).all()) or not bool(torch.isfinite(feature).all())  # 执行当前语句以推进本节示例。
                or own.device != self.update_message.weight.device or other.device != own.device  # 计算并保存当前步骤的中间状态。
                or feature.device != own.device or own.dtype != self.update_message.weight.dtype  # 计算并保存当前步骤的中间状态。
                or other.dtype != own.dtype or feature.dtype != own.dtype  # 计算并保存当前步骤的中间状态。
                or isinstance(delta, bool) or not isinstance(delta, (int, float))  # 执行当前语句以推进本节示例。
                or not math.isfinite(float(delta)) or delta < 0):  # 执行当前语句以推进本节示例。
            raise ValueError("更新消息输入 shape/数值非法")  # 遇到非法合同立即显式失败。
        time = self.update_time(torch.tensor([delta], dtype=torch.float32)).squeeze(0)  # 计算并保存当前步骤的中间状态。
        result = torch.tanh(self.update_message(torch.cat([own, other, feature, time])))  # 计算并保存当前步骤的中间状态。
        if not bool(torch.isfinite(result).all()): raise ValueError("更新消息产生非有限值")  # 按当前条件选择后续控制路径。
        return result  # 返回当前分支计算出的结果。

    def propose_updates(self, bank: MemoryBank, event: Event) -> tuple[torch.Tensor, torch.Tensor]:  # 定义本节可复用的核心函数。
        self._validate_bank(bank); validate_event50(event)  # 执行当前语句以推进本节示例。
        src_memory = bank.read(event.src, event.key)  # 计算并保存当前步骤的中间状态。
        dst_memory = bank.read(event.dst, event.key)  # 计算并保存当前步骤的中间状态。
        src_last = bank.last_key[event.src]  # 计算并保存当前步骤的中间状态。
        dst_last = bank.last_key[event.dst]  # 计算并保存当前步骤的中间状态。
        src_delta = float(event.timestamp - src_last[0]) if src_last is not None else float(event.timestamp)  # 计算并保存当前步骤的中间状态。
        dst_delta = float(event.timestamp - dst_last[0]) if dst_last is not None else float(event.timestamp)  # 计算并保存当前步骤的中间状态。
        feature = torch.tensor(event.feature, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        src_input = self._update_input(src_memory, dst_memory, feature, src_delta)  # 计算并保存当前步骤的中间状态。
        dst_input = self._update_input(dst_memory, src_memory, feature, dst_delta)  # 计算并保存当前步骤的中间状态。
        new_src = self.updater(src_input, src_memory)  # 计算并保存当前步骤的中间状态。
        new_dst = self.updater(dst_input, dst_memory)  # 计算并保存当前步骤的中间状态。
        return new_src, new_dst  # 返回当前分支计算出的结果。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def update_after(self, bank: MemoryBank, event: Event) -> None:  # 定义本节可复用的核心函数。
        self._validate_bank(bank); validate_event50(event)  # 执行当前语句以推进本节示例。
        src_memory = bank.read(event.src, event.key)  # 计算并保存当前步骤的中间状态。
        dst_memory = bank.read(event.dst, event.key)  # 计算并保存当前步骤的中间状态。
        feature = torch.tensor(event.feature, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        new_src, new_dst = self.propose_updates(bank, event)  # 计算并保存当前步骤的中间状态。
        bank.commit_many([  # 执行当前语句以推进本节示例。
            (event.src, event.key, new_src, Mail(event.key, dst_memory, feature)),  # 执行当前语句以推进本节示例。
            (event.dst, event.key, new_dst, Mail(event.key, src_memory, feature)),  # 执行当前语句以推进本节示例。
        ])  # 执行当前语句以推进本节示例。

torch.manual_seed(5002)  # 执行当前语句以推进本节示例。
model_probe50 = TGNLinkPredictor(NUM_NODES)  # 计算并保存当前步骤的中间状态。
cold_bank50 = MemoryBank(NUM_NODES, 8)  # 计算并保存当前步骤的中间状态。
cold_score50 = model_probe50(cold_bank50, 0, 8, (1, 0))  # 计算并保存当前步骤的中间状态。
assert cold_score50.ndim == 0 and torch.isfinite(cold_score50)  # 用受控断言验证关键不变量。
model_probe50.update_after(cold_bank50, events50[0])  # 执行当前语句以推进本节示例。
assert not cold_bank50.memory.requires_grad  # 用受控断言验证关键不变量。
assert all(not mail.neighbor_memory.requires_grad for box in cold_bank50.mailbox for mail in box)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    model_probe50(cold_bank50, events50[0].src, events50[0].dst, events50[0].key)  # 执行当前语句以推进本节示例。
    raise AssertionError("update-before-predict 泄漏未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "先预测后更新" in str(exc)  # 用受控断言验证关键不变量。

cold_bank50.reset()  # 执行当前语句以推进本节示例。
assert torch.equal(cold_bank50.memory, torch.zeros_like(cold_bank50.memory))  # 用受控断言验证关键不变量。
assert all(key is None for key in cold_bank50.last_key)  # 用受控断言验证关键不变量。
assert all(len(box) == 0 for box in cold_bank50.mailbox)  # 用受控断言验证关键不变量。

for bad_query50 in ((8, 0, (2, 0)), (0, 12, (2, 0))):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        model_probe50(cold_bank50, *bad_query50)  # 执行当前语句以推进本节示例。
        raise AssertionError("非法 schema/未激活查询未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    model_probe50.score_memory_states(0, 8, torch.full((8,), float("nan")), torch.zeros(8))  # 执行当前语句以推进本节示例。
    raise AssertionError("非有限 score override 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "非法" in str(exc) or "有限" in str(exc)  # 用受控断言验证关键不变量。


## 6. 同 timestamp 与 batch 内更新顺序

所谓“batch”不能默认同时写 memory：若两条同 timestamp 事件共享节点，`sequence=1` 应看到 `sequence=0` 的 mailbox。下面记录每次预测前的 mailbox 长度，证明更新发生在单条预测之后、下一条预测之前。


In [ ]:
same_time50 = [  # 计算并保存当前步骤的中间状态。
    Event("same-0", 0, 8, 5, 0, (1.0, 0.0), "train"),  # 执行当前语句以推进本节示例。
    Event("same-1", 0, 9, 5, 1, (1.0, 1.0), "train"),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
validate_event_stream(same_time50)  # 执行当前语句以推进本节示例。
order_bank50 = MemoryBank(NUM_NODES, 8)  # 计算并保存当前步骤的中间状态。
before_counts50, order_scores50 = [], []  # 计算并保存当前步骤的中间状态。
for event in same_time50:  # 遍历输入元素以累积或检查结果。
    before_counts50.append(len(order_bank50.context(event.src, event.key)))  # 执行当前语句以推进本节示例。
    order_scores50.append(float(model_probe50(order_bank50, event.src, event.dst, event.key)))  # 执行当前语句以推进本节示例。
    model_probe50.update_after(order_bank50, event)  # 执行当前语句以推进本节示例。
assert before_counts50 == [0, 1]  # 用受控断言验证关键不变量。
assert order_bank50.last_key[0] == (5, 1)  # 用受控断言验证关键不变量。
assert all(math.isfinite(score) for score in order_scores50)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    model_probe50.update_after(order_bank50, same_time50[0])  # 执行当前语句以推进本节示例。
    raise AssertionError("旧 sequence 重放未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "先预测后更新" in str(exc) or "递增" in str(exc)  # 用受控断言验证关键不变量。


## 7. 时间感知负采样与 Filtered 候选

查询 $(u,?,t)$ 时，候选 item 必须已在 $t$ 激活；从候选中移除 `key<=query_key` 的其他已知真边，但绝不能使用未来真值做过滤。评估时保留当前正例，训练负采样再从其余候选中取样。

未来会成为真边的 item 在早期仍可作为负候选，这是 prequential 协议的现实含义；如果业务不能容忍这种标签漂移，需要另行定义观察窗，而不是偷看未来全集。


In [ ]:
def truth_as_of(events: list[Event], src: int, query_key: tuple[int, int]) -> set[int]:  # 定义本节可复用的核心函数。
    validate_key50(query_key, "query_key"); validate_event_stream(events)  # 执行当前语句以推进本节示例。
    if not isinstance(src, int) or isinstance(src, bool) or src not in USER_IDS:  # 按当前条件选择后续控制路径。
        raise ValueError("src 必须是 user")  # 遇到非法合同立即显式失败。
    return {e.dst for e in events if e.src == src and e.key <= query_key}  # 返回当前分支计算出的结果。

def filtered_candidates(events: list[Event], src: int, true_dst: int,  # 定义本节可复用的核心函数。
                        query_key: tuple[int, int]) -> list[int]:  # 执行当前语句以推进本节示例。
    validate_query50(src, true_dst, query_key)  # 执行当前语句以推进本节示例。
    active = [item for item in ITEM_IDS if ACTIVE_FROM[item] <= query_key[0]]  # 计算并保存当前步骤的中间状态。
    known = truth_as_of(events, src, query_key)  # 计算并保存当前步骤的中间状态。
    result = [item for item in active if item == true_dst or item not in known]  # 计算并保存当前步骤的中间状态。
    if true_dst not in result:  # 按当前条件选择后续控制路径。
        raise ValueError("当前正例未进入 filtered 候选")  # 遇到非法合同立即显式失败。
    return result  # 返回当前分支计算出的结果。

def sample_negative(events: list[Event], event: Event, rng: random.Random) -> int:  # 定义本节可复用的核心函数。
    validate_event50(event)  # 执行当前语句以推进本节示例。
    if not isinstance(rng, random.Random): raise ValueError("rng 必须是独立 random.Random")  # 按当前条件选择后续控制路径。
    candidates = [item for item in filtered_candidates(events, event.src, event.dst, event.key) if item != event.dst]  # 计算并保存当前步骤的中间状态。
    if not candidates: raise ValueError("没有合法时间感知负样本")  # 按当前条件选择后续控制路径。
    return candidates[rng.randrange(len(candidates))]  # 返回当前分支计算出的结果。

history_probe50 = [  # 计算并保存当前步骤的中间状态。
    Event("past", 0, 8, 1, 0, (1.0, 0.0), "train"),  # 执行当前语句以推进本节示例。
    Event("future", 0, 9, 3, 0, (1.0, 0.0), "val"),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
early_candidates50 = filtered_candidates(history_probe50, 0, 10, (2, 0))  # 计算并保存当前步骤的中间状态。
late_candidates50 = filtered_candidates(history_probe50, 0, 8, (3, 1))  # 计算并保存当前步骤的中间状态。
assert 9 in early_candidates50  # 未来真边没有提前用于过滤
assert 8 not in early_candidates50  # 用受控断言验证关键不变量。
assert 9 not in late_candidates50 and 8 in late_candidates50  # 用受控断言验证关键不变量。
assert 12 not in filtered_candidates(events50, 0, 8, (16, 1))  # 用受控断言验证关键不变量。
assert sample_negative(events50, events50[0], random.Random(7)) in {9, 10, 11}  # 用受控断言验证关键不变量。


## 8. 受控训练：每轮 reset，逐事件先预测后更新

每轮从空 memory 重放 train 流：先对正例和一个时间合法负例做在线打分；再生成**尚未提交**的可微更新状态，并增加权重 0.20 的一步 post-event 对比辅助项，使手写 updater 确实获得梯度；最后才在 `no_grad` 下把状态 detach 后提交。这样不会跨整条日志保留 autograd 图，也不会把当前事件提前泄漏给在线分数。长程截断 BPTT 仍属于生产训练设计，本 fixture 不声称覆盖。


In [ ]:
torch.manual_seed(5003)  # 执行当前语句以推进本节示例。
model50 = TGNLinkPredictor(NUM_NODES)  # 计算并保存当前步骤的中间状态。
optimizer50 = torch.optim.Adam(model50.parameters(), lr=0.035)  # 计算并保存当前步骤的中间状态。
loss_trace50 = []  # 计算并保存当前步骤的中间状态。
for epoch in range(12):  # 遍历输入元素以累积或检查结果。
    model50.train(); bank50 = MemoryBank(NUM_NODES, 8); optimizer50.zero_grad()  # 计算并保存当前步骤的中间状态。
    losses = []  # 计算并保存当前步骤的中间状态。
    rng = random.Random(SEED + epoch)  # 计算并保存当前步骤的中间状态。
    for event in split_events50["train"]:  # 遍历输入元素以累积或检查结果。
        negative = sample_negative(events50, event, rng)  # 计算并保存当前步骤的中间状态。
        positive_logit = model50(bank50, event.src, event.dst, event.key)  # 计算并保存当前步骤的中间状态。
        negative_logit = model50(bank50, event.src, negative, event.key)  # 计算并保存当前步骤的中间状态。
        proposed_src, proposed_dst = model50.propose_updates(bank50, event)  # 计算并保存当前步骤的中间状态。
        negative_memory = bank50.read(negative, event.key)  # 计算并保存当前步骤的中间状态。
        post_positive = model50.score_memory_states(event.src, event.dst, proposed_src, proposed_dst)  # 计算并保存当前步骤的中间状态。
        post_negative = model50.score_memory_states(event.src, negative, proposed_src, negative_memory)  # 计算并保存当前步骤的中间状态。
        online_loss = F.softplus(-positive_logit) + F.softplus(negative_logit)  # 计算并保存当前步骤的中间状态。
        updater_aux = F.softplus(-post_positive) + F.softplus(post_negative)  # 计算并保存当前步骤的中间状态。
        losses.append(online_loss + 0.20 * updater_aux)  # 执行当前语句以推进本节示例。
        model50.update_after(bank50, event)  # 执行当前语句以推进本节示例。
    epoch_loss = torch.stack(losses).mean()  # 计算并保存当前步骤的中间状态。
    epoch_loss.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model50.parameters(), 5.0)  # 执行当前语句以推进本节示例。
    optimizer50.step(); loss_trace50.append(float(epoch_loss.detach()))  # 执行当前语句以推进本节示例。

assert len(loss_trace50) == 12  # 用受控断言验证关键不变量。
assert loss_trace50[-1] < loss_trace50[0] * 0.85  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model50.parameters())  # 用受控断言验证关键不变量。
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model50.updater.parameters())  # 用受控断言验证关键不变量。
assert all(torch.isfinite(p).all() for p in model50.parameters())  # 用受控断言验证关键不变量。


## 9. Prequential filtered ranking

评估某个 split 前，只重放更早 split；对每条当前事件：先给全部时间合法 filtered candidates 打分，计算真例 rank，再写入当前事件。模型在 test 前冻结，test 内仍按真实在线顺序更新 memory，但绝不更新参数。

报告 MRR 和 Hits@1。这里每个 user 周期性连接固定 item，属于可记忆 fixture；高分只验证在线时序和候选协议。


In [ ]:
def prequential_metrics(model, prefix: list[Event], query_events: list[Event], all_events: list[Event]):  # 定义本节可复用的核心函数。
    model.eval(); bank = MemoryBank(NUM_NODES, model.memory_dim)  # 计算并保存当前步骤的中间状态。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        for event in prefix:  # 遍历输入元素以累积或检查结果。
            model.update_after(bank, event)  # 执行当前语句以推进本节示例。
        ranks = []  # 计算并保存当前步骤的中间状态。
        for event in query_events:  # 遍历输入元素以累积或检查结果。
            candidates = filtered_candidates(all_events, event.src, event.dst, event.key)  # 计算并保存当前步骤的中间状态。
            scores = torch.stack([model(bank, event.src, item, event.key) for item in candidates])  # 计算并保存当前步骤的中间状态。
            true_index = candidates.index(event.dst)  # 计算并保存当前步骤的中间状态。
            rank = 1 + int((scores > scores[true_index]).sum())  # 计算并保存当前步骤的中间状态。
            ranks.append(rank)  # 执行当前语句以推进本节示例。
            model.update_after(bank, event)  # 执行当前语句以推进本节示例。
    return {"mrr": sum(1.0 / r for r in ranks) / len(ranks),  # 返回当前分支计算出的结果。
            "hits1": sum(r == 1 for r in ranks) / len(ranks), "ranks": tuple(ranks)}  # 计算并保存当前步骤的中间状态。

val_metrics50 = prequential_metrics(model50, split_events50["train"], split_events50["val"], events50)  # 计算并保存当前步骤的中间状态。
test_prefix50 = split_events50["train"] + split_events50["val"]  # 计算并保存当前步骤的中间状态。
test_metrics50 = prequential_metrics(model50, test_prefix50, split_events50["test"], events50)  # 计算并保存当前步骤的中间状态。
assert len(val_metrics50["ranks"]) == 8 and len(test_metrics50["ranks"]) == 8  # 用受控断言验证关键不变量。
assert val_metrics50["hits1"] >= 0.75  # 用受控断言验证关键不变量。
assert test_metrics50["hits1"] >= 0.75  # 用受控断言验证关键不变量。
assert 0.0 < test_metrics50["mrr"] <= 1.0  # 用受控断言验证关键不变量。


## 10. 制品、会话化发布服务与可恢复在线状态

制品 manifest 不只保存网络维度，还绑定 node/type registry、`active_from`、完整事件快照、时间 split、同 timestamp 排序规则、mailbox 容量、负采样/filtered 语义、梯度裁剪及 reset/detach recipe。权重摘要逐项覆盖 key、dtype、shape、bytes。loader 返回 `PublishedTGNService`，不暴露一个需要调用方自行管理 MemoryBank 的裸模型。

每个 `session_id` 独占 MemoryBank、offset、全局 watermark 与已处理 event ID 集合。`score_then_commit` 在深拷贝上完成“打分→双端点更新”，全部成功后一次替换会话状态；重放 ID、相同/倒退 key 和半提交都 fail-closed。快照显式携带上述状态、release 与 digest；digest 用于发现传输损坏，release 的真实性仍由包外 publisher registry 保证。

包内摘要可被攻击者重算，所以包外只读 publisher registry 才是权重制品的信任锚。整体替换权重并重新生成内部摘要仍必须失败。


In [ ]:
def event_record50(event: Event) -> dict:  # 定义本节可复用的核心函数。
    validate_event50(event)  # 执行当前语句以推进本节示例。
    return {"id": event.event_id, "src": event.src, "dst": event.dst, "timestamp": event.timestamp,  # 返回当前分支计算出的结果。
            "sequence": event.sequence, "feature": list(event.feature), "split": event.split}  # 执行当前语句以推进本节示例。

EVENT_SNAPSHOT50 = canonical_digest([event_record50(e) for e in events50])  # 计算并保存当前步骤的中间状态。
RELEASE50 = "tgn-demo-50/v2"  # 计算并保存当前步骤的中间状态。
SNAPSHOT_VERSION50 = 1  # 计算并保存当前步骤的中间状态。
MANIFEST50 = {  # 计算并保存当前步骤的中间状态。
    "config": {"num_nodes": NUM_NODES, "memory_dim": 8, "feature_dim": 2, "time_dim": 4},  # 执行当前语句以推进本节示例。
    "schema": {"users": list(USER_IDS), "items": list(ITEM_IDS), "active_from": ACTIVE_FROM},  # 执行当前语句以推进本节示例。
    "event_snapshot": EVENT_SNAPSHOT50,  # 执行当前语句以推进本节示例。
    "split": {s: [e.event_id for e in split_events50[s]] for s in ("train", "val", "test")},  # 执行当前语句以推进本节示例。
    "recipe": {"seed": SEED, "epochs": 12, "lr": 0.035, "grad_clip": 5.0,  # 执行当前语句以推进本节示例。
               "order": "(timestamp,sequence)", "protocol": "predict-then-update",  # 执行当前语句以推进本节示例。
               "mailbox_size": 4, "negative_filter": "truth_as_of_query_key",  # 执行当前语句以推进本节示例。
               "loss": "online_pairwise+0.20*post_event_updater_aux",  # 执行当前语句以推进本节示例。
               "memory": "reset_each_epoch+detach_each_commit"},  # 执行当前语句以推进本节示例。
    "serving": {"state": "per_session", "atomicity": "per-session-lock+clone-score-update-swap",  # 执行当前语句以推进本节示例。
                "snapshot_version": SNAPSHOT_VERSION50,  # 执行当前语句以推进本节示例。
                "idempotency": "event_id+strict_global_watermark"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
state50 = {k: v.detach().cpu().clone() for k, v in model50.state_dict().items()}  # 计算并保存当前步骤的中间状态。
package50 = {"release_id": RELEASE50, "manifest": copy.deepcopy(MANIFEST50), "state": state50}  # 计算并保存当前步骤的中间状态。
package50["state_digest"] = state_digest50(state50)  # 计算并保存当前步骤的中间状态。
package50["package_digest"] = canonical_digest({"release_id": RELEASE50, "manifest": package50["manifest"],  # 计算并保存当前步骤的中间状态。
                                                  "state_digest": package50["state_digest"]})  # 执行当前语句以推进本节示例。
_PUBLISHER_REGISTRY50 = MappingProxyType({RELEASE50: package50["package_digest"]})  # 计算并保存当前步骤的中间状态。

@dataclass  # 为下方定义附加声明式配置。
class _SessionState50:  # 定义承载本节状态与行为的数据结构。
    bank: MemoryBank  # 执行当前语句以推进本节示例。
    offset: int  # 执行当前语句以推进本节示例。
    watermark: tuple[int, int] | None  # 执行当前语句以推进本节示例。
    processed_ids: set[str]  # 执行当前语句以推进本节示例。

class PublishedTGNService:  # 定义承载本节状态与行为的数据结构。
    _SNAPSHOT_FIELDS = {"version", "release_id", "session_id", "offset", "watermark",  # 计算并保存当前步骤的中间状态。
                        "processed_event_ids", "bank", "snapshot_digest"}  # 执行当前语句以推进本节示例。

    def __init__(self, model: TGNLinkPredictor, release_id: str, manifest: dict):  # 定义本节可复用的核心函数。
        if not isinstance(model, TGNLinkPredictor) or not isinstance(release_id, str):  # 按当前条件选择后续控制路径。
            raise ValueError("发布服务配置非法")  # 遇到非法合同立即显式失败。
        self._model, self.release_id, self._manifest = model.eval(), release_id, copy.deepcopy(manifest)  # 计算并保存当前步骤的中间状态。
        for parameter in self._model.parameters(): parameter.requires_grad_(False)  # 遍历输入元素以累积或检查结果。
        self._sessions: dict[str, _SessionState50] = {}  # 计算并保存当前步骤的中间状态。
        self._registry_lock = threading.RLock()  # 计算并保存当前步骤的中间状态。
        self._session_locks: dict[str, threading.RLock] = {}  # 计算并保存当前步骤的中间状态。

    @property  # 为下方定义附加声明式配置。
    def manifest(self) -> dict:  # 定义本节可复用的核心函数。
        return copy.deepcopy(self._manifest)  # 返回当前分支计算出的结果。

    @staticmethod  # 为下方定义附加声明式配置。
    def _session_id(session_id: str) -> str:  # 定义本节可复用的核心函数。
        if (not isinstance(session_id, str) or not session_id or session_id != session_id.strip()  # 按当前条件选择后续控制路径。
                or len(session_id) > 128):  # 执行当前语句以推进本节示例。
            raise ValueError("session_id 必须是 1..128 字符的无首尾空白字符串")  # 遇到非法合同立即显式失败。
        return session_id  # 返回当前分支计算出的结果。

    def _lock_for(self, session_id: str):  # 定义本节可复用的核心函数。
        with self._registry_lock:  # 在受管理的上下文中执行操作。
            if session_id not in self._session_locks:  # 按当前条件选择后续控制路径。
                self._session_locks[session_id] = threading.RLock()  # 计算并保存当前步骤的中间状态。
            return self._session_locks[session_id]  # 返回当前分支计算出的结果。

    def _blank_state(self) -> _SessionState50:  # 定义本节可复用的核心函数。
        cfg, mailbox_size = self._manifest["config"], self._manifest["recipe"]["mailbox_size"]  # 计算并保存当前步骤的中间状态。
        bank = MemoryBank(cfg["num_nodes"], cfg["memory_dim"], cfg["feature_dim"], mailbox_size)  # 计算并保存当前步骤的中间状态。
        return _SessionState50(bank, 0, None, set())  # 返回当前分支计算出的结果。

    def session_status(self, session_id: str) -> dict:  # 定义本节可复用的核心函数。
        session_id = self._session_id(session_id)  # 计算并保存当前步骤的中间状态。
        with self._lock_for(session_id):  # 在受管理的上下文中执行操作。
            state = self._sessions.get(session_id)  # 计算并保存当前步骤的中间状态。
            return {"exists": state is not None, "offset": 0 if state is None else state.offset,  # 返回当前分支计算出的结果。
                    "watermark": None if state is None else state.watermark,  # 执行当前语句以推进本节示例。
                    "processed_count": 0 if state is None else len(state.processed_ids)}  # 执行当前语句以推进本节示例。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def score(self, session_id: str, src: int, dst: int, query_key: tuple[int, int]) -> float:  # 定义本节可复用的核心函数。
        session_id = self._session_id(session_id); key = validate_query50(src, dst, query_key)  # 计算并保存当前步骤的中间状态。
        with self._lock_for(session_id):  # 在受管理的上下文中执行操作。
            state = self._sessions.get(session_id)  # 计算并保存当前步骤的中间状态。
            if state is not None and state.watermark is not None and key <= state.watermark:  # 按当前条件选择后续控制路径。
                raise ValueError("query_key 不得落在会话 watermark 之前或之上")  # 遇到非法合同立即显式失败。
            bank = self._blank_state().bank if state is None else state.bank  # 计算并保存当前步骤的中间状态。
            value = float(self._model(bank, src, dst, key))  # 计算并保存当前步骤的中间状态。
            if not math.isfinite(value): raise ValueError("服务打分非有限")  # 按当前条件选择后续控制路径。
            return value  # 返回当前分支计算出的结果。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def score_then_commit(self, session_id: str, event: Event) -> float:  # 定义本节可复用的核心函数。
        session_id = self._session_id(session_id); key = validate_event50(event)  # 计算并保存当前步骤的中间状态。
        with self._lock_for(session_id):  # 在受管理的上下文中执行操作。
            previous = self._sessions.get(session_id)  # 计算并保存当前步骤的中间状态。
            base = self._blank_state() if previous is None else previous  # 计算并保存当前步骤的中间状态。
            if event.event_id in base.processed_ids:  # 按当前条件选择后续控制路径。
                raise ValueError("重复 event_id 被拒绝")  # 遇到非法合同立即显式失败。
            if base.watermark is not None and key <= base.watermark:  # 按当前条件选择后续控制路径。
                raise ValueError("事件 key 未严格越过会话 watermark")  # 遇到非法合同立即显式失败。
            # 锁内在私有副本执行；异常不污染状态，并发请求也不能丢失更新。
            working_bank = base.bank.clone()  # 计算并保存当前步骤的中间状态。
            value = float(self._model(working_bank, event.src, event.dst, key))  # 计算并保存当前步骤的中间状态。
            if not math.isfinite(value): raise ValueError("服务打分非有限")  # 按当前条件选择后续控制路径。
            self._model.update_after(working_bank, event)  # 执行当前语句以推进本节示例。
            processed = set(base.processed_ids); processed.add(event.event_id)  # 计算并保存当前步骤的中间状态。
            self._sessions[session_id] = _SessionState50(working_bank, base.offset + 1, key, processed)  # 计算并保存当前步骤的中间状态。
            return value  # 返回当前分支计算出的结果。

    def snapshot(self, session_id: str) -> dict:  # 定义本节可复用的核心函数。
        session_id = self._session_id(session_id)  # 计算并保存当前步骤的中间状态。
        with self._lock_for(session_id):  # 在受管理的上下文中执行操作。
            if session_id not in self._sessions: raise ValueError("未知 session，不能生成快照")  # 按当前条件选择后续控制路径。
            state = self._sessions[session_id]  # 计算并保存当前步骤的中间状态。
            body = {"version": SNAPSHOT_VERSION50, "release_id": self.release_id,  # 计算并保存当前步骤的中间状态。
                    "session_id": session_id, "offset": state.offset,  # 执行当前语句以推进本节示例。
                    "watermark": None if state.watermark is None else list(state.watermark),  # 执行当前语句以推进本节示例。
                    "processed_event_ids": sorted(state.processed_ids), "bank": state.bank.state_payload()}  # 执行当前语句以推进本节示例。
            return {**body, "snapshot_digest": canonical_digest(body)}  # 返回当前分支计算出的结果。

    def restore(self, snapshot: dict) -> None:  # 定义本节可复用的核心函数。
        if not isinstance(snapshot, dict) or set(snapshot) != self._SNAPSHOT_FIELDS:  # 按当前条件选择后续控制路径。
            raise ValueError("服务快照字段集合非法")  # 遇到非法合同立即显式失败。
        body = {key: copy.deepcopy(value) for key, value in snapshot.items() if key != "snapshot_digest"}  # 计算并保存当前步骤的中间状态。
        if not isinstance(snapshot["snapshot_digest"], str) or canonical_digest(body) != snapshot["snapshot_digest"]:  # 按当前条件选择后续控制路径。
            raise ValueError("服务快照 digest 不匹配")  # 遇到非法合同立即显式失败。
        if body["version"] != SNAPSHOT_VERSION50 or body["release_id"] != self.release_id:  # 按当前条件选择后续控制路径。
            raise ValueError("快照版本或 release 不匹配")  # 遇到非法合同立即显式失败。
        session_id = self._session_id(body["session_id"])  # 计算并保存当前步骤的中间状态。
        offset, raw_ids, raw_watermark = body["offset"], body["processed_event_ids"], body["watermark"]  # 计算并保存当前步骤的中间状态。
        if (not isinstance(offset, int) or isinstance(offset, bool) or offset < 0  # 按当前条件选择后续控制路径。
                or not isinstance(raw_ids, list) or any(not isinstance(value, str) or not value for value in raw_ids)  # 执行当前语句以推进本节示例。
                or len(raw_ids) != len(set(raw_ids)) or len(raw_ids) != offset):  # 计算并保存当前步骤的中间状态。
            raise ValueError("快照 offset/processed_event_ids 合同非法")  # 遇到非法合同立即显式失败。
        if raw_watermark is None:  # 按当前条件选择后续控制路径。
            watermark = None  # 计算并保存当前步骤的中间状态。
        elif isinstance(raw_watermark, list):  # 按当前条件选择后续控制路径。
            watermark = validate_key50(tuple(raw_watermark), "snapshot watermark")  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            raise ValueError("snapshot watermark 类型非法")  # 遇到非法合同立即显式失败。
        bank = MemoryBank.from_payload(body["bank"])  # 计算并保存当前步骤的中间状态。
        cfg = self._manifest["config"]  # 计算并保存当前步骤的中间状态。
        expected_bank_config = {"num_nodes": cfg["num_nodes"], "memory_dim": cfg["memory_dim"],  # 计算并保存当前步骤的中间状态。
                                "feature_dim": cfg["feature_dim"],  # 执行当前语句以推进本节示例。
                                "mailbox_size": self._manifest["recipe"]["mailbox_size"]}  # 执行当前语句以推进本节示例。
        if bank.state_payload()["config"] != expected_bank_config:  # 按当前条件选择后续控制路径。
            raise ValueError("快照 MemoryBank 与 release 配置不匹配")  # 遇到非法合同立即显式失败。
        node_watermarks = [key for key in bank.last_key if key is not None]  # 计算并保存当前步骤的中间状态。
        if offset == 0:  # 按当前条件选择后续控制路径。
            if watermark is not None or node_watermarks:  # 按当前条件选择后续控制路径。
                raise ValueError("空 offset 快照却含状态")  # 遇到非法合同立即显式失败。
        elif (watermark is None or not node_watermarks or max(node_watermarks) != watermark  # 按当前条件选择后续控制路径。
                or any(key > watermark for key in node_watermarks)):  # 执行当前语句以推进本节示例。
            raise ValueError("快照 watermark 与节点状态不一致")  # 遇到非法合同立即显式失败。
        with self._lock_for(session_id):  # 在受管理的上下文中执行操作。
            if session_id in self._sessions: raise ValueError("拒绝覆盖已存在 session")  # 按当前条件选择后续控制路径。
            self._sessions[session_id] = _SessionState50(bank, offset, watermark, set(raw_ids))  # 计算并保存当前步骤的中间状态。

def load_published_tgn(package: dict) -> PublishedTGNService:  # 定义本节可复用的核心函数。
    required = {"release_id", "manifest", "state", "state_digest", "package_digest"}  # 计算并保存当前步骤的中间状态。
    if not isinstance(package, dict) or set(package) != required:  # 按当前条件选择后续控制路径。
        raise ValueError("package 字段集合非法")  # 遇到非法合同立即显式失败。
    release_id = package["release_id"]  # 计算并保存当前步骤的中间状态。
    if release_id not in _PUBLISHER_REGISTRY50: raise ValueError("未知 release")  # 按当前条件选择后续控制路径。
    actual_state = state_digest50(package["state"])  # 计算并保存当前步骤的中间状态。
    actual_package = canonical_digest({"release_id": release_id, "manifest": package["manifest"],  # 计算并保存当前步骤的中间状态。
                                       "state_digest": actual_state})  # 执行当前语句以推进本节示例。
    if actual_state != package["state_digest"] or actual_package != package["package_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("包内摘要不匹配")  # 遇到非法合同立即显式失败。
    if actual_package != _PUBLISHER_REGISTRY50[release_id]:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry 信任锚不匹配")  # 遇到非法合同立即显式失败。
    if package["manifest"] != MANIFEST50:  # 按当前条件选择后续控制路径。
        raise ValueError("schema/event/split/recipe/serving 合同不匹配")  # 遇到非法合同立即显式失败。
    restored_model = TGNLinkPredictor(**package["manifest"]["config"])  # 计算并保存当前步骤的中间状态。
    restored_model.load_state_dict(package["state"], strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedTGNService(restored_model, release_id, package["manifest"])  # 返回当前分支计算出的结果。

restored50 = load_published_tgn(package50)  # 计算并保存当前步骤的中间状态。
fresh_bank50 = MemoryBank(NUM_NODES, 8)  # 计算并保存当前步骤的中间状态。
assert math.isclose(restored50.score("read-only", 0, 8, (1, 0)),  # 用受控断言验证关键不变量。
                    float(model50(fresh_bank50, 0, 8, (1, 0))), rel_tol=0.0, abs_tol=1e-7)  # 计算并保存当前步骤的中间状态。
assert not restored50.session_status("read-only")["exists"]  # 纯 score 不创建状态
assert package50["manifest"]["event_snapshot"] == EVENT_SNAPSHOT50  # 用受控断言验证关键不变量。
assert package50["manifest"]["recipe"]["grad_clip"] == 5.0  # 用受控断言验证关键不变量。
manifest_copy50 = restored50.manifest  # 计算并保存当前步骤的中间状态。
manifest_copy50["recipe"]["mailbox_size"] = 999  # 计算并保存当前步骤的中间状态。
assert restored50.manifest["recipe"]["mailbox_size"] == 4  # 用受控断言验证关键不变量。

# 对外读出的 mailbox 是深拷贝，调用方修改它不会反向污染 bank。
clone_probe_bank50 = MemoryBank(NUM_NODES, 8)  # 计算并保存当前步骤的中间状态。
model50.update_after(clone_probe_bank50, events50[0])  # 执行当前语句以推进本节示例。
mail_copy50 = clone_probe_bank50.context(events50[0].src, (1, 1))[0]  # 计算并保存当前步骤的中间状态。
mail_copy50.neighbor_memory.add_(999.0); mail_copy50.feature.fill_(999.0)  # 执行当前语句以推进本节示例。
mail_again50 = clone_probe_bank50.context(events50[0].src, (1, 1))[0]  # 计算并保存当前步骤的中间状态。
assert not torch.equal(mail_copy50.neighbor_memory, mail_again50.neighbor_memory)  # 用受控断言验证关键不变量。
assert not torch.equal(mail_copy50.feature, mail_again50.feature)  # 用受控断言验证关键不变量。

# 会话 A 的提交不创建或污染会话 B；拒绝重放后快照逐字节语义不变。
restored50.score_then_commit("tenant-A", events50[0])  # 执行当前语句以推进本节示例。
snapshot_a_before50 = restored50.snapshot("tenant-A")  # 计算并保存当前步骤的中间状态。
blank_b_score50 = restored50.score("tenant-B", 0, 8, (1, 1))  # 计算并保存当前步骤的中间状态。
blank_reference50 = float(model50(MemoryBank(NUM_NODES, 8), 0, 8, (1, 1)))  # 计算并保存当前步骤的中间状态。
assert math.isclose(blank_b_score50, blank_reference50, rel_tol=0.0, abs_tol=1e-7)  # 用受控断言验证关键不变量。
assert restored50.session_status("tenant-A")["offset"] == 1  # 用受控断言验证关键不变量。
assert restored50.session_status("tenant-B")["offset"] == 0  # 用受控断言验证关键不变量。
assert any(any(value != 0.0 for value in row) for row in snapshot_a_before50["bank"]["memory"])  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    restored50.score_then_commit("tenant-A", events50[0])  # 执行当前语句以推进本节示例。
    raise AssertionError("重复 event_id 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "重复" in str(exc)  # 用受控断言验证关键不变量。
assert restored50.snapshot("tenant-A")["snapshot_digest"] == snapshot_a_before50["snapshot_digest"]  # 用受控断言验证关键不变量。

concurrent_service50 = load_published_tgn(package50)  # 计算并保存当前步骤的中间状态。
concurrent_outcomes50 = []  # 计算并保存当前步骤的中间状态。
def concurrent_submit50():  # 定义本节可复用的核心函数。
    try:  # 尝试执行可能失败的受控操作。
        concurrent_service50.score_then_commit("same-session", events50[0])  # 执行当前语句以推进本节示例。
        concurrent_outcomes50.append("committed")  # 执行当前语句以推进本节示例。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        concurrent_outcomes50.append("duplicate" if "重复" in str(exc) else "unexpected")  # 执行当前语句以推进本节示例。
threads50 = [threading.Thread(target=concurrent_submit50) for _ in range(2)]  # 计算并保存当前步骤的中间状态。
for thread in threads50: thread.start()  # 遍历输入元素以累积或检查结果。
for thread in threads50: thread.join(timeout=5.0)  # 遍历输入元素以累积或检查结果。
assert all(not thread.is_alive() for thread in threads50)  # 用受控断言验证关键不变量。
assert sorted(concurrent_outcomes50) == ["committed", "duplicate"]  # 用受控断言验证关键不变量。
assert concurrent_service50.session_status("same-session")["offset"] == 1  # 用受控断言验证关键不变量。

stale_event50 = Event("stale-key", 0, 8, 1, 0, (1.0, 0.0), "train")  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    restored50.score_then_commit("tenant-A", stale_event50)  # 执行当前语句以推进本节示例。
    raise AssertionError("相同 watermark 的不同 ID 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "watermark" in str(exc)  # 用受控断言验证关键不变量。
assert restored50.snapshot("tenant-A")["snapshot_digest"] == snapshot_a_before50["snapshot_digest"]  # 用受控断言验证关键不变量。

# 快照恢复后继续处理同一事件，分数与下一快照完全一致。
resume_service50 = load_published_tgn(package50)  # 计算并保存当前步骤的中间状态。
for event in events50[:2]: resume_service50.score_then_commit("resume-50", event)  # 遍历输入元素以累积或检查结果。
resume_snapshot50 = resume_service50.snapshot("resume-50")  # 计算并保存当前步骤的中间状态。
assert resume_snapshot50["offset"] == 2 and resume_snapshot50["watermark"] == [1, 1]  # 用受控断言验证关键不变量。
assert resume_snapshot50["processed_event_ids"] == sorted(e.event_id for e in events50[:2])  # 用受控断言验证关键不变量。
recovered_service50 = load_published_tgn(package50)  # 计算并保存当前步骤的中间状态。
recovered_service50.restore(copy.deepcopy(resume_snapshot50))  # 执行当前语句以推进本节示例。
next_original50 = resume_service50.score_then_commit("resume-50", events50[2])  # 计算并保存当前步骤的中间状态。
next_recovered50 = recovered_service50.score_then_commit("resume-50", events50[2])  # 计算并保存当前步骤的中间状态。
assert math.isclose(next_original50, next_recovered50, rel_tol=0.0, abs_tol=1e-7)  # 用受控断言验证关键不变量。
assert resume_service50.snapshot("resume-50")["snapshot_digest"] == recovered_service50.snapshot("resume-50")["snapshot_digest"]  # 用受控断言验证关键不变量。

tampered_snapshot50 = copy.deepcopy(resume_snapshot50)  # 计算并保存当前步骤的中间状态。
tampered_snapshot50["offset"] += 1  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_tgn(package50).restore(tampered_snapshot50)  # 执行当前语句以推进本节示例。
    raise AssertionError("digest 不匹配的快照被接受")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "digest" in str(exc)  # 用受控断言验证关键不变量。

resigned_bad_snapshot50 = copy.deepcopy(resume_snapshot50)  # 计算并保存当前步骤的中间状态。
resigned_bad_snapshot50["bank"]["memory"][0][0] = float("nan")  # 计算并保存当前步骤的中间状态。
snapshot_body50 = {key: value for key, value in resigned_bad_snapshot50.items() if key != "snapshot_digest"}  # 计算并保存当前步骤的中间状态。
resigned_bad_snapshot50["snapshot_digest"] = canonical_digest(snapshot_body50)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_tgn(package50).restore(resigned_bad_snapshot50)  # 执行当前语句以推进本节示例。
    raise AssertionError("重算 digest 的非有限快照被接受")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "数值" in str(exc) or "有限" in str(exc)  # 用受控断言验证关键不变量。

# 攻击者整体替换权重并重算所有包内摘要，仍过不了包外 registry。
forged50 = copy.deepcopy(package50)  # 计算并保存当前步骤的中间状态。
first_key50 = next(iter(forged50["state"]))  # 计算并保存当前步骤的中间状态。
forged50["state"][first_key50] = forged50["state"][first_key50] + 0.01  # 计算并保存当前步骤的中间状态。
forged50["state_digest"] = state_digest50(forged50["state"])  # 计算并保存当前步骤的中间状态。
forged50["package_digest"] = canonical_digest({"release_id": RELEASE50, "manifest": forged50["manifest"],  # 计算并保存当前步骤的中间状态。
                                                 "state_digest": forged50["state_digest"]})  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_published_tgn(forged50)  # 执行当前语句以推进本节示例。
    raise AssertionError("重算所有内部摘要的伪造包被接受")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(exc)  # 用受控断言验证关键不变量。


## 11. 失败模式与生产差距

- **时间泄漏**：先 update 后 score；同 timestamp 没有 sequence；filtered ranking 使用全量未来真值；邻居查询返回查询时刻之后的 memory。
- **状态污染**：训练 epoch/用户会话之间没 reset；不同 session 共用 bank；memory 保存 autograd 图；负候选逐个打分时偷偷改变状态；离线回放与线上排序规则不一致。这里用 per-session `RLock` 与 clone-then-swap 演示进程内原子性。
- **快照边界**：普通 SHA-256 digest 能发现损坏，却不能证明调用方自行重签的快照可信；跨信任域要用 KMS/HMAC/数字签名。恢复还应配合追加日志、保留策略和密钥轮换。
- **生产系统**：需要迟到/乱序事件水位线、幂等 event ID、分片 memory store、checkpoint+日志恢复、热点节点并发控制、TTL/GDPR 删除、漂移监控。多进程/多机不能依赖 Python 锁，必须用带版本号的 CAS 或存储事务。
- **训练差距**：真实 TGN 会设计截断 BPTT、批内同节点冲突处理、大规模时间邻居采样和 calibration；本例把协议正确性置于 benchmark 分数之前。


In [ ]:
assert events50[0].key < events50[-1].key  # 用受控断言验证关键不变量。
assert val_metrics50["mrr"] >= val_metrics50["hits1"]  # 用受控断言验证关键不变量。
assert test_metrics50["mrr"] >= test_metrics50["hits1"]  # 用受控断言验证关键不变量。
assert package50["package_digest"] == _PUBLISHER_REGISTRY50[RELEASE50]  # 用受控断言验证关键不变量。
assert set(MANIFEST50["split"]) == {"train", "val", "test"}  # 用受控断言验证关键不变量。
